# Stage 2: PDF extraction

Turns the handbook PDF into 254 articles made of typed elements: headings, text (paragraphs and list items),
tables, code and images. The code is `src/handbook_qa/pdf_extract.py`; this notebook runs it and checks the result.

Re-running: set `REBUILD = True` to extract from the PDF again (about 75 s). Otherwise the saved
`data/build/articles.json` is reused if it exists.

In [1]:
import collections
import re
import time

import pandas as pd
import pymupdf

from handbook_qa.config import load_settings
from handbook_qa.ingest import download
from handbook_qa.pdf_extract import (
    _page_lines,
    extract_articles,
    load_articles,
    save_articles,
)

# Settings and display options
REBUILD = False
settings = load_settings()
paths = settings.paths
pd.set_option("display.max_colwidth", 140)

In [2]:
download(settings)  # downloads the PDF if missing and checks its SHA-256
# Extract or load the articles
if REBUILD or not paths.articles.exists():
    started = time.perf_counter()
    articles = extract_articles(paths.pdf)
    save_articles(articles, paths.articles)
    print(f"extracted in {time.perf_counter() - started:.0f}s")
else:
    articles = load_articles(paths.articles)
    print("loaded saved", paths.articles.relative_to(paths.data.parent))
print(len(articles), "articles")

loaded saved data\build\articles.json
254 articles


## What came out

In [3]:
# Flatten articles into elements
elements = [(a, e) for a in articles for e in a.elements]
kinds = collections.Counter(e.kind for _, e in elements)
levels = collections.Counter(f"H{e.level}" for _, e in elements if e.kind == "heading")
# Element, heading and title counts
print("elements by kind:", dict(kinds))
print("headings by level:", dict(sorted(levels.items())))
print("articles without a title:", sum(not a.title for a in articles))
print("unique images:", len({e.xref for _, e in elements if e.kind == "image"}),
      "| image placements:", kinds["image"])

elements by kind: {'text': 12964, 'heading': 2656, 'image': 78, 'code': 103, 'table': 150}
headings by level: {'H1': 23, 'H2': 1107, 'H3': 1214, 'H4': 312}
articles without a title: 0
unique images: 77 | image placements: 78


## The browser header and footer are gone

Every page carries 4 lines from Chrome's print (date, title, `file://` URL, page number). They are dropped by font
and position; none should be left in the text.

In [4]:
# Look for leftover header and footer text
all_text = "\n".join(e.text for _, e in elements)
print("file:// URLs left:", all_text.count("file:///tmp/handbook-pdf-build"))
print("'N/1076' page numbers left:", len(re.findall(r"\b\d{1,4}/1076\b", all_text)))

file:// URLs left: 0
'N/1076' page numbers left: 0


## Rebuilding paragraphs and lists

Text blocks in this PDF are not paragraphs: on page 1 one block holds three list items, and bullets are not text.
A line continues the previous one only if the gap is a normal line gap *and* the previous line ran out of room.
Below: the raw lines of page 1 with their geometry, then the elements built from them.

In [5]:
# Load the raw lines of page 1
with pymupdf.open(paths.pdf) as doc:
    raw = _page_lines(doc[0])
# Show their geometry
pd.DataFrame([{"x0": round(l.x0), "x1": round(l.x1), "right_edge": round(l.right_edge), "y0": round(l.y0),
               "text": l.text[:90]} for l in raw[3:21]])

,x0,x1,right_edge,y0,text
0,64,531,531,165,🎨 Need artwork or merch? Please request it using the request templates. Do not request art
1,64,206,531,182,or merch over Slack or email.
2,50,373,558,209,"All artwork and merch requests are handled by , , and on the team."
3,50,219,558,236,They can help you with things like:
4,80,264,548,263,Custom visuals for paid ad campaigns
5,80,227,548,279,Blog and social media artwork
6,80,196,548,294,New themed hedgehogs
7,80,209,548,311,Custom CTAs and banners
8,80,155,548,327,Branded merch
9,80,190,548,342,Animated UI elements


In [6]:
# Show elements built from article 1
art_requests = articles[0]
pd.DataFrame([{"kind": e.kind, "page": e.page, "level": e.level or "", "text": e.text[:130]}
              for e in art_requests.elements[:22]])

,kind,page,level,text
0,text,1,,🎨 Need artwork or merch? Please request it using the request templates. Do not request art or merch over Slack or email.
1,text,1,,"All artwork and merch requests are handled by , , and on the team."
2,text,1,,They can help you with things like:
3,text,1,,- Custom visuals for paid ad campaigns
4,text,1,,- Blog and social media artwork
5,text,1,,- New themed hedgehogs
6,text,1,,- Custom CTAs and banners
7,text,1,,- Branded merch
8,text,1,,- Animated UI elements
9,text,1,,"They get a lot of work requests, so they use two separate project boards to organize work – one for merch and one for other art pr"


Two cases the geometry alone gets wrong, and the rules added for them (both have unit tests):
- **List items that end near the right edge.** "...move the task forward." ends at x=550 and the next item starts
  with "No", which would not have fitted either. Inside lists, a sentence end followed by a capital starts a new item.
- **Hyphen line breaks.** Chrome breaks after hyphens ("right-" / "hand"); those words are joined without a space.

In [7]:
# Print the two problem cases
for _, e in elements:
    if "Feedback/Review for 10+" in e.text or "right-hand sidebar" in e.text:
        print(repr(e.text[:110]))

'  - Feedback/Review for 10+ days: asks if any feedback is needed to move the task forward.'
"2. Select the hog you want. If needed, adjust the size using the 'Frame' menu in the top of the right-hand sid"
"3. At the bottom of the right-hand sidebar, select the file type you need in the 'Export' menu, choose @2x, th"
"2. Select the hog you want. If needed, adjust the size using the 'Frame' menu in the top of the right-hand sid"
"3. At the bottom of the right-hand sidebar, select the file type you need in the 'Export' menu, choose @2x, th"
'Important: Always attach incident-related tickets to the open incident using the incident.io app in the right-'


## Tables and code

Tables are found with PyMuPDF's table finder (only on pages with 10 pt table text) and kept whole as markdown.
Code blocks are lines set entirely in Menlo inside the code box (x≈61); indentation is restored from x positions.

In [8]:
# Count tables and rows
tables = [(a, e) for a, e in elements if e.kind == "table"]
print(len(tables), "tables; rows per table:", collections.Counter(e.text.count("\n") - 1 for _, e in tables).most_common(5))
# Show the eligibility table
a, e = next((a, e) for a, e in tables if "Eligibility" in e.text)
print(f"\n{a.path}, page {e.page}:\n{e.text}")

150 tables; rows per table: [(4, 33), (5, 30), (3, 24), (2, 24), (1, 15)]

contents/handbook/brand/startups.md, page 20:
|Feature|Startups|YCombinator|
|---|---|---|
|Eligibility|<2 years old, <$5M raised, not acquired|Must be in YC, <$25m raised|
|Credit|$50,000 for 12 months|$50k per year, whilst eligible|
|Can use credit for add-ons?|⚠ Yes, but cannot use credit for BAA in Boost add-on|✅ Yes, and can use credit for BAA in Boost add-on|
|Founder merch|Welcome pack (max 1)|Different welcome pack (max 4)|
|Community|—|Tim's Whatsapp, priority support|
|Apply via…|Startup page|Secret YC page|


In [9]:
# Count code blocks
codes = [(a, e) for a, e in elements if e.kind == "code"]
print(len(codes), "code blocks")
# Show one code block
a, e = codes[3]
print(f"{a.path}, page {e.page}:\n{e.text[:500]}")

103 code blocks
contents/handbook/company/goal-setting.md, page 67:
<details>
<summary>Write the goal here (owner)</summary>
- **Rationale:** Why is this important?
- **What we'll ship:** Keep this brief
- **We'll know we're successful when:** Metric or outcome
</details>


## Known extraction problems

1. **Blank names and teams** (216, see notebook 01): left as they are, by decision 07.
2. **Dollar amounts rendered as maths.** Some `$...$` pairs in the source were rendered by the site's maths
   renderer, which drops spaces. A rough way to find them: very long runs of letters with no spaces.

In [10]:
# Find suspiciously long letter runs
mangled = [(a.path, e.page, m.group(0)) for a, e in elements if e.kind == "text"
           for m in re.finditer(r"[A-Za-z′']{30,}", e.text)]
# Count them and show the first ten
print(len(mangled), "suspicious runs")
pd.DataFrame(mangled, columns=["article", "page", "text"]).head(10)

2 suspicious runs


,article,page,text
0,contents/handbook/brand/startups.md,21,theyonlyneedtoregisteronceanditwillrenewautomaticallywhilethey′reeligible
1,contents/handbook/growth/sales/how-to-do-discovery.md,593,Ourteamlooksaftercustomerswhoarepayingorcouldpay
